# SCOPE P2 — ML Risk Research Notebook

This notebook trains and evaluates a public **Ethereum address-reputation** classifier. It does **not** directly classify SCOPE action payloads. The dataset is extremely imbalanced, so accuracy alone is not sufficient and no 99% score is promised. The deterministic SCOPE policy remains authoritative; ML must never override a hard DENY.

**Run instructions:** In VS Code/Jupyter, choose a Python environment, then use **Run All**. The dataset downloads automatically. The notebook saves the trained artifact and evaluation report under `artifacts/` and `reports/`.

In [ ]:
# Cell 1 — install notebook dependencies (safe to rerun)
%pip -q install pandas numpy scikit-learn joblib matplotlib


In [ ]:
# Cell 2 — imports and workspace paths
from pathlib import Path
import json, urllib.request, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, balanced_accuracy_score, f1_score,
                             precision_score, recall_score)
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

SEED = 42
ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "raw"
ARTIFACT_DIR = ROOT / "artifacts"
REPORT_DIR = ROOT / "reports"
for folder in (DATA_DIR, ARTIFACT_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "BLTE_multiclass_v2.csv"
MODEL_PATH = ARTIFACT_DIR / "scope_address_risk_model.joblib"
METRICS_PATH = REPORT_DIR / "scope_ml_evaluation.json"
print("Workspace:", ROOT)
print("Dataset path:", DATA_PATH)
print("Seed:", SEED)


In [ ]:
# Cell 3 — download the public dataset (no account/access request required)
DATA_URL = "https://raw.githubusercontent.com/salam-ammari/Graph-Neural-Network-Based-Fraudulent-Address-Classification-in-Ethereum-Network/main/BLTE_multiclass_v2.csv"
if not DATA_PATH.exists() or DATA_PATH.stat().st_size < 100_000:
    request = urllib.request.Request(DATA_URL, headers={"User-Agent": "SCOPE-P2-ML-Notebook/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response, open(DATA_PATH, "wb") as output_file:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            output_file.write(chunk)
print(f"Dataset downloaded/available: {DATA_PATH} ({DATA_PATH.stat().st_size / (1024*1024):.1f} MiB)")


In [ ]:
# Cell 4 — inspect schema and class balance
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = [str(column).strip() for column in df.columns]
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

# This dataset's readable multiclass target is named 'attack'.
# 'attack_encoded' and 'Scam' are not input features; they may leak target information.
if "attack" not in df.columns:
    raise ValueError("Expected target column 'attack' was not found. Available columns: " + str(df.columns.tolist()))
label_col = "attack"
print("Target column:", label_col)
print("Class counts:")
print(df[label_col].value_counts(dropna=False))
print("Class percentages:")
display((df[label_col].value_counts(normalize=True, dropna=False) * 100).round(4).rename("percent"))


In [ ]:
# Cell 5 — prepare features and encode the TEXT labels correctly
# Important: attack contains names such as Normal, Phishing, Scamming—not numeric 0/1/2.
label_map = {"Normal": 0, "Phishing": 1, "Scamming": 2}
labels_text = df[label_col].astype(str).str.strip()
valid = labels_text.isin(label_map)
if not valid.all():
    print("Rows skipped for unknown/missing labels:", int((~valid).sum()))
df_model = df.loc[valid].copy().reset_index(drop=True)
y = labels_text.loc[valid].map(label_map).astype(int).reset_index(drop=True)

# Remove target, numeric target encoding, address identifiers, and the Scam flag to prevent leakage.
exclude = {"attack", "attack_encoded", "Scam", "Address"}
id_tokens = ("address", "wallet", "account", "hash", "txid", "node_id")
drop_cols = set(exclude)
for column in df_model.columns:
    if any(token in column.lower() for token in id_tokens):
        drop_cols.add(column)
X = df_model.drop(columns=[c for c in drop_cols if c in df_model.columns], errors="ignore")
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna(axis=1, how="all")
X = X.loc[:, X.nunique(dropna=True) > 1].copy()
X = X.fillna(X.median(numeric_only=True)).fillna(0).reset_index(drop=True)

if len(X) != len(y) or X.shape[1] < 2 or y.nunique() != 3:
    raise ValueError(f"Invalid feature matrix: X={X.shape}, y={y.shape}, classes={y.nunique()}")
print("Feature matrix:", X.shape)
print("Class counts (0=Normal, 1=Phishing, 2=Scamming):", y.value_counts().sort_index().to_dict())
print("Feature columns:", X.columns.tolist())


In [ ]:
# Cell 6 — stratified 60/20/20 train/validation/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)
print({"train": len(X_train), "validation": len(X_val), "test": len(X_test)})
print("Test class counts:", y_test.value_counts().sort_index().to_dict())


In [ ]:
# Cell 7 — train baselines and choose using validation macro-F1
# Macro-F1 gives each class equal weight; it is more informative than accuracy for this imbalanced dataset.
candidates = {
    "Dummy-most-frequent": DummyClassifier(strategy="most_frequent"),
    "RandomForest-balanced": RandomForestClassifier(
        n_estimators=160, min_samples_leaf=2, class_weight="balanced_subsample",
        max_features="sqrt", random_state=SEED, n_jobs=-1
    ),
    "ExtraTrees-balanced": ExtraTreesClassifier(
        n_estimators=160, min_samples_leaf=2, class_weight="balanced",
        max_features="sqrt", random_state=SEED, n_jobs=-1
    )
}
fitted = {}
validation_rows = []
for name, estimator in candidates.items():
    pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", estimator)])
    start = time.time()
    pipeline.fit(X_train, y_train)
    val_pred = pipeline.predict(X_val)
    score = f1_score(y_val, val_pred, average="macro", zero_division=0)
    fitted[name] = pipeline
    validation_rows.append({"model": name, "validation_macro_f1": score, "seconds": round(time.time() - start, 2)})
validation_table = pd.DataFrame(validation_rows).sort_values("validation_macro_f1", ascending=False)
display(validation_table)
best_name = str(validation_table.iloc[0]["model"])
best_model = fitted[best_name]
print("Selected using validation macro-F1:", best_name)


In [ ]:
# Cell 8 — final held-out test evaluation (test data was not used to select the model)
test_pred = best_model.predict(X_test)
class_names = {0: "NORMAL", 1: "PHISHING", 2: "SCAMMING"}
present_labels = [0, 1, 2]
report = classification_report(
    y_test, test_pred, labels=present_labels,
    target_names=[class_names[v] for v in present_labels],
    output_dict=True, zero_division=0
)
metrics = {
    "selected_model": best_name,
    "accuracy": float(accuracy_score(y_test, test_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, test_pred)),
    "macro_precision": float(precision_score(y_test, test_pred, average="macro", zero_division=0)),
    "macro_recall": float(recall_score(y_test, test_pred, average="macro", zero_division=0)),
    "macro_f1": float(f1_score(y_test, test_pred, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(y_test, test_pred, average="weighted", zero_division=0)),
    "classification_report": report
}
display(pd.DataFrame({k: v for k, v in metrics.items() if k != "classification_report"}, index=[0]).T)
print(json.dumps(metrics, indent=2))


In [ ]:
# Cell 9 — confusion matrix and class-wise report
cm = confusion_matrix(y_test, test_pred, labels=present_labels)
display(pd.DataFrame(report).T)
display(pd.DataFrame(cm, index=[class_names[i] for i in present_labels], columns=[class_names[i] for i in present_labels]))
ConfusionMatrixDisplay(cm, display_labels=[class_names[i] for i in present_labels]).plot(values_format="d", xticks_rotation=25)
plt.title(f"Held-out test confusion matrix — {best_name}")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 10 — refit on train+validation and export model + evaluation report
# Test metrics above remain the held-out evaluation for model selection.
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)
best_model.fit(X_trainval, y_trainval)
bundle = {
    "model_version": "scope-address-risk-v1",
    "prediction_type": "address_reputation",
    "model": best_model,
    "feature_names": list(map(str, X.columns)),
    "label_names": {"0": "Normal", "1": "Phishing", "2": "Scamming"},
    "dataset_name": "BLTE multiclass Ethereum address-risk benchmark",
    "seed": SEED,
    "warning": "Address-level model; not a direct SCOPE action-level classifier."
}
joblib.dump(bundle, MODEL_PATH)
result = {
    "model_version": bundle["model_version"],
    "dataset": bundle["dataset_name"],
    "rows_used": int(len(X)),
    "feature_count": int(X.shape[1]),
    "class_counts": {str(int(k)): int(v) for k, v in y.value_counts().sort_index().items()},
    "split_sizes": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "metrics": metrics,
    "confusion_matrix_labels": present_labels,
    "confusion_matrix": cm.tolist(),
    "limitations": [
        "Dataset labels Ethereum addresses, not individual transaction actions.",
        "Random stratified split is not temporal or external validation.",
        "Severe class imbalance makes accuracy alone misleading.",
        "ML is advisory; deterministic hard-deny policy remains authoritative."
    ]
}
METRICS_PATH.write_text(json.dumps(result, indent=2), encoding="utf-8")
print("Saved model:", MODEL_PATH)
print("Saved metrics:", METRICS_PATH)


In [ ]:
# Cell 11 — verify saved artifact can be loaded
loaded = joblib.load(MODEL_PATH)
assert loaded["model_version"] == "scope-address-risk-v1"
assert loaded["prediction_type"] == "address_reputation"
assert len(loaded["feature_names"]) == X.shape[1]
smoke_prediction = loaded["model"].predict(X_test.iloc[:5])
print("Artifact load and prediction smoke test: PASS")
print("Model version:", loaded["model_version"])
print("Feature count:", len(loaded["feature_names"]))
print("Sample predictions:", [loaded["label_names"][str(int(v))] for v in smoke_prediction])
print("Evaluation report:", METRICS_PATH)


## Integration boundary
- The exported model predicts **address reputation** from the dataset's behavioral features.
- It cannot be connected directly to SCOPE action fields unless a trusted address-feature provider supplies the exact expected columns.
- Do not fabricate missing features or train/test on the deterministic policy's own labels.
- Keep explicit DENY rules and consent requirements in the deterministic policy layer; ML must never override them.
- Report held-out macro-F1, per-class recall, confusion matrix, and accuracy. Do not claim 99% unless the measured results support that claim and external validation confirms it.